# Brain-state transitions vs. anxiety and depression

Two figures, both on Cam-CAN HADS (anxiety and depression categories):

1. **Four rankings side by side.** `bstm_selection` (transition matrices) and
   `fcm_selection` (static connectivity) at rest and during the movie. One faint
   dot per state set or atlas, the best one highlighted with its spread over fold
   seeds. The dashed tick is the covariates-only baseline, which every row has to beat.
2. **Volcano, bstm only.** For rest and movie, the `TOP_N` best state sets from
   figure 1. Every transition cell is tested against HADS with a partial Spearman
   correlation (adjusted for `COVARIATES`). Colour shows the state set, and the
   legend rank (#1…) is the set's position in that panel's ranking.

The selection runs (`fmri-decomp select-bstm` / `select-fcm`) produce the
rankings. This notebook only reads them, plus the `transitions/` tables, and writes nothing to `outputs/`.
Everything you might want to change is in the next cell.

In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
CAMCAN = Path(os.environ.get("CAMCAN_PHENO",
                             "/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                             "opendata/paule_toussaint_camcan01870"))
PHENO_FILES = [(CAMCAN / "approved_data.tsv", "\t"),   # HADS
               (CAMCAN / "standard_data.csv", ",")]     # Age, Sex
PHENO_ID = "CCID"

# label -> column. Labels are what the figures say.
TARGETS = {"anxiety":    "additional_HADS_anx_category",
           "depression": "additional_HADS_dep_category"}
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]   # low -> high

# task label in the *_selection trees -> task value inside the transition tables
TASKS = {"rest": "rest", "movie": "movie"}                  # matched case-insensitively
COHORTS = ["camcan", "camcan_rest"]                         # cohort= dirs to read

MODEL = "ridge"                                  # ridge | hgb | lgbm | logistic
ARM = {"bstm": "cells [joint]", "fcm": "edges"}  # the hypothesis arm per tree
BASELINE_ARM = "(covariates only)"

TOP_N = 10                                       # state sets per volcano panel
COVARIATES = ["Age", "Sex", "n_transitions"]
CATEGORICAL = ["Sex"]
ALPHA = 0.05                                     # BH-FDR, per panel
LABEL_TOP = 3                                    # cells annotated per panel
SAVE_DIR = None                                  # e.g. Path("figures/08") to save PNG + SVG

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#a3a29d", "#fcfcfb"
TREE_COLOUR = {"bstm": "#2a78d6", "fcm": "#eb6834"}
# 8 validated hues, then two neutrals for slots 9-10. The marker shape is the
# second encoding, so no two sets rely on colour alone.
SET_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
               "#008300", "#6250d6", "#e34948", "#52514e", "#8c6d4f"]
SET_MARKERS = ["o", "s", "^", "D", "v"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "axes.titlesize": 11,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 9, "legend.frameon": False})

def hive(path, key):
    """One `key=value` partition from a path, by name."""
    return next((p.split("=", 1)[1] for p in Path(path).parts if p.startswith(f"{key}=")), None)

def fmt_window(w):
    return str(int(w)) if float(w).is_integer() else str(w)

def set_label(r):
    """A readable name for one summary row: state set or static-FC atlas."""
    if r.get("tree") == "fcm":
        return r["atlas"]
    w = "1 TR" if float(r["window_s"]) < 0 else f"{fmt_window(r['window_s'])}s"
    return f"{r['states']} · {r['atlas']} · {w}"

def save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        for ext in ("png", "svg"):
            fig.savefig(Path(SAVE_DIR) / f"{name}.{ext}", dpi=200, bbox_inches="tight")

## 1. The four rankings

In [ ]:
def read_summaries(tree):
    files = sorted((ROOT / f"{tree}_selection").glob("task=*/target=*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv under {ROOT / f'{tree}_selection'}")
    return pd.concat([pd.read_csv(p).assign(tree=tree, task=hive(p, "task"),
                                            target=hive(p, "target"))
                      for p in files], ignore_index=True)

label_of = {col: lab for lab, col in TARGETS.items()}
summ = pd.concat([read_summaries("bstm"), read_summaries("fcm")], ignore_index=True)
summ = summ[(summ["model"] == MODEL) & summ["target"].isin(label_of)
            & summ["task"].isin(TASKS)].assign(phenotype=lambda d: d["target"].map(label_of))
summ["label"] = summ.apply(set_label, axis=1)

# Check this before reading a gap as a result: the runs are separate, so `n` can differ.
summ.pivot_table(index=["tree", "task"], columns="phenotype", values="n", aggfunc="max")

In [ ]:
RUNS = [(task, tree) for task in TASKS for tree in ("bstm", "fcm")]

def plot_runs(ax, d):
    rng = np.random.default_rng(0)
    for i, (task, tree) in enumerate(RUNS):
        g = d[(d["task"] == task) & (d["tree"] == tree)]
        hyp, base = g[g["arm"] == ARM[tree]], g[g["arm"] == BASELINE_ARM]
        if hyp.empty:
            continue
        ax.scatter(i + rng.uniform(-.14, .14, len(hyp)), hyp["mean"], s=12,
                   color=TREE_COLOUR[tree], alpha=.3, lw=0)
        if len(base):
            ax.hlines(base["mean"].mean(), i - .3, i + .3, color=INK2, ls="--", lw=1)
        best = hyp.loc[hyp["mean"].idxmax()]
        ax.errorbar(i, best["mean"], yerr=best["std"], fmt="o", ms=8, lw=2,
                    color=TREE_COLOUR[tree], mec=SURFACE, mew=1.5)
        ax.annotate(best["label"].replace(" · ", "\n", 1), (i, best["mean"]),
                    xytext=(10, 0), textcoords="offset points",
                    fontsize=7, color=INK2, va="center")
    ax.set_xticks(range(len(RUNS)), [f"{tree}\n{task}" for task, tree in RUNS])
    for k in range(1, len(TASKS)):
        ax.axvline(2 * k - .5, color=MUTED, lw=.6)
    ax.axhline(0, color=MUTED, lw=.6)
    ax.set_xlim(-.6, len(RUNS) - .1)

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 4.2), sharey=True)
for ax, (lab, d) in zip(np.atleast_1d(axes), summ.groupby("phenotype", sort=False)):
    plot_runs(ax, d)
    ax.set_title(lab)
np.atleast_1d(axes)[0].set_ylabel(f"{MODEL} CV score ({'AUC' if MODEL == 'logistic' else 'Spearman'})")
fig.suptitle(f"bstm {ARM['bstm']} vs fcm {ARM['fcm']} · large dot = best ± sd over seeds · "
             "dashed = covariates only", fontsize=8, color=INK2, y=.99)
fig.tight_layout()
save(fig, "fig1_rankings")

## 2. Volcano: transition cells of the best state sets

There is one panel for each (task, phenotype) pair. Each panel draws its `TOP_N` sets from its own ranking,
so the colours in different panels belong to different sets: read the legend under each panel.
Dots are `(state set, i->j)` cells, and filled dots with a dark ring survive BH-FDR over every cell in the panel.
The dotted line marks p = 0.05 uncorrected.

The cells are compositional, because each subject's row sums to 1, so hits tend to come in sprays.
Read the strongest cell and treat its neighbours as the same effect.

In [ ]:
def read_pheno():
    frames = [pd.read_csv(p, sep=s, dtype={PHENO_ID: str}) for p, s in PHENO_FILES]
    ph = frames[0]
    for f in frames[1:]:
        ph = ph.merge(f.drop(columns=[c for c in f if c in ph and c != PHENO_ID]),
                      on=PHENO_ID, how="outer")
    lower = {c.lower(): c for c in ph.columns}              # Age vs age
    ph = ph.rename(columns={lower[c.lower()]: c for c in COVARIATES if c.lower() in lower})
    ph["sub"] = ph[PHENO_ID].astype(str).str.strip().str.upper()
    code = {lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)}
    for lab, col in TARGETS.items():
        ph[lab] = ph[col].astype("string").str.strip().map(code).astype(float)
    for c in set(COVARIATES) - set(CATEGORICAL) - {"n_transitions"}:
        ph[c] = pd.to_numeric(ph[c], errors="coerce")
    return ph[["sub", *TARGETS, *(c for c in COVARIATES if c in ph)]]

def load_set(atlas, window_s, states, task):
    """One state set's transition table for one task, one row per subject."""
    base = (ROOT / "transitions" / f"atlas={atlas}" / f"window_s={fmt_window(window_s)}"
            / f"states={states}")
    files = [p for p in base.rglob("subjects.parquet")
             if hive(p, "cohort") in COHORTS
             and (hive(p, "task") or task).lower() == TASKS[task].lower()]
    if not files:
        raise SystemExit(f"no transition table for {base} task={task}")
    d = pd.concat(map(pd.read_parquet, files), ignore_index=True)
    d = d[d["task"].astype(str).str.lower() == TASKS[task].lower()]
    d["sub"] = d["sub"].astype(str).str.strip().str.upper()
    if d["sub"].duplicated().any():
        raise SystemExit(f"{states} {task}: a subject appears twice -- narrow COHORTS")
    return d

def cell_cols(d):
    return [c for c in d.columns if "->" in c]

def as_p_cond(d, K):
    """p_joint -> Markov p(j | i): back to counts, then renormalise each row."""
    cols = cell_cols(d)
    M = d[cols].to_numpy(float).reshape(len(d), K, K) * d["n_transitions"].to_numpy(float)[:, None, None]
    s = M.sum(2, keepdims=True)
    return pd.DataFrame(np.divide(M, s, out=np.zeros_like(M), where=s > 0).reshape(len(d), -1),
                        columns=cols, index=d.index)

def partial_spearman(A, y, cov):
    """Rank everything, residualise on the covariates, correlate. Returns r, p."""
    X = pd.get_dummies(cov, columns=[c for c in CATEGORICAL if c in cov],
                       drop_first=True).astype(float)
    X = np.column_stack([np.ones(len(X)), X.fillna(X.mean()).to_numpy()])
    R = np.column_stack([stats.rankdata(y), np.apply_along_axis(stats.rankdata, 0, A)])
    R = R - X @ np.linalg.lstsq(X, R, rcond=None)[0]
    R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), np.nan)
    r = R[:, 1:].T @ R[:, 0] / len(R)
    df = len(R) - X.shape[1] - 1
    p = 2 * stats.t.sf(np.abs(r * np.sqrt(df / (1 - r ** 2))), df)
    return r, p

pheno = read_pheno()
print(f"phenotype: {len(pheno):,} subjects;",
      ", ".join(f"{lab} {pheno[lab].value_counts().sort_index().astype(int).to_dict()}"
                for lab in TARGETS))

In [ ]:
def top_sets(task, lab, n=TOP_N):
    g = summ[(summ["tree"] == "bstm") & (summ["task"] == task) & (summ["phenotype"] == lab)
             & (summ["arm"] == ARM["bstm"])]
    return (g.sort_values("mean", ascending=False)
             .drop_duplicates(["atlas", "window_s", "states"]).head(n).reset_index(drop=True))

def scan(task, lab):
    """Every transition cell of the top sets vs one phenotype. One row per cell."""
    p_norm = "cond" if "[cond]" in ARM["bstm"] else "joint"
    out = []
    for rank, s in top_sets(task, lab).iterrows():
        t = load_set(s["atlas"], s["window_s"], s["states"], task)
        if p_norm == "cond":
            t[cell_cols(t)] = as_p_cond(t, int(s["K"]))
        d = t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
            subset=[lab, *(c for c in COVARIATES if c not in CATEGORICAL)])
        A = d[cell_cols(t)]
        A = A.loc[:, A.std() > 0]                         # flat cells are untestable
        r, p = partial_spearman(A.to_numpy(float), d[lab].to_numpy(float), d[COVARIATES])
        out.append(pd.DataFrame({"rank": rank + 1, "set": s["label"], "score": s["mean"],
                                 "transition": A.columns, "r": r, "p": p, "n": len(d)}))
    v = pd.concat(out, ignore_index=True)
    v["p_fdr"] = stats.false_discovery_control(v["p"], method="bh")
    return v.assign(task=task, phenotype=lab)

volc = pd.concat([scan(task, lab) for task in TASKS for lab in TARGETS], ignore_index=True)
volc.nsmallest(10, "p")[["task", "phenotype", "rank", "set", "transition", "r", "p", "p_fdr", "n"]]

In [ ]:
def plot_volcano(ax, v):
    sig = v["p_fdr"] < ALPHA
    for k, (rank, g) in enumerate(v.groupby("rank")):
        c, m = SET_COLOURS[k % len(SET_COLOURS)], SET_MARKERS[k % len(SET_MARKERS)]
        y = -np.log10(g["p"])
        ax.scatter(g["r"], y, s=14, marker=m, color=c, alpha=.45, lw=0,
                   label=f"#{rank}  {g['set'].iloc[0]}")
        hit = sig[g.index]
        ax.scatter(g["r"][hit], y[hit], s=40, marker=m, color=c, edgecolor=INK, lw=.8)
    for t in v.nsmallest(LABEL_TOP, "p").itertuples():
        ax.annotate(f"{t.transition} (#{t.rank})", (t.r, -np.log10(t.p)), xytext=(4, 3),
                    textcoords="offset points", fontsize=7, color=INK2)
    ax.axhline(-np.log10(.05), color=MUTED, ls=":", lw=.8)
    if sig.any():
        ax.axhline(-np.log10(v.loc[sig, "p"].max()), color=INK2, ls="--", lw=.8)
    ax.axvline(0, color=MUTED, lw=.6)
    lim = 1.08 * v["r"].abs().max()
    ax.set_xlim(-lim, lim)
    ax.set_title(f"{v['task'].iloc[0]} · {v['phenotype'].iloc[0]}   "
                 f"({sig.sum()} / {len(v):,} cells FDR<{ALPHA:g}, n≈{int(v['n'].median())})",
                 fontsize=10)
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.16), ncol=2, fontsize=7,
              handletextpad=.3, columnspacing=1, markerscale=1.3)

fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                         figsize=(6.5 * len(TARGETS), 6.3 * len(TASKS)))
for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
    plot_volcano(axes[i, j], volc[(volc["task"] == task) & (volc["phenotype"] == lab)])
for ax in axes[:, 0]:
    ax.set_ylabel(r"$-\log_{10}$ p")
for ax in axes[-1]:
    ax.set_xlabel(f"partial Spearman r  (| {', '.join(COVARIATES)})")
fig.tight_layout(h_pad=3)
save(fig, "fig2_volcano")

### Editing notes

* **Another model or arm:** change `MODEL` / `ARM`. With `ARM["bstm"] = "cells [cond]"`, the volcano also tests
  the Markov p(j | i) instead of the stored p_joint.
* **More or fewer sets:** `TOP_N`. Past ten, colours repeat with a new marker shape.
* **A single panel:** call `plot_volcano(ax, volc[(volc.task == "rest") & (volc.phenotype == "anxiety")])`.
* **Hits for one set:** `volc.query("rank == 1 and phenotype == 'anxiety'").nsmallest(10, "p")`.
* `top_sets(task, phenotype)` shows exactly which rows of `summary.csv` the volcano drew from.
* The ranking score and the volcano answer different questions. A set can rank first because its signal
  is spread across many cells and still show no single cell above the FDR line.